# Day 29 — Django URL Configuration Repair

This notebook consolidates the URL configuration from the current project state you shared.

It:
- finds the repository root,
- verifies all referenced view modules exist before making changes,
- creates timestamped backups,
- registers one dashboard route and one prediction-centre route,
- runs `manage.py check`, resolves expected URLs, and runs the existing pagination tests.

**Important:** This notebook does not add health or ingestion routes without verified views. Run it from your repository root and inspect each output. It changes URL files only after the preflight checks pass.


In [1]:
from pathlib import Path
import os
import shutil
import datetime

# Find the repository root from the current directory or its parents.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "django_api" / "manage.py").exists()
     and (p / "django_api" / "config" / "urls.py").exists()
     and (p / "django_api" / "ipl_api" / "urls.py").exists()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find django_api/manage.py and both URL files. "
        "Change the notebook working directory to the repository root and rerun."
    )

DJANGO_ROOT = PROJECT_ROOT / "django_api"
CONFIG_URLS = DJANGO_ROOT / "config" / "urls.py"
APP_URLS = DJANGO_ROOT / "ipl_api" / "urls.py"

print("Project root:", PROJECT_ROOT)
print("Django root:", DJANGO_ROOT)
print("Config URL file exists:", CONFIG_URLS.exists())
print("App URL file exists:", APP_URLS.exists())


Project root: D:\Git\Personal\ipl-player-performance
Django root: D:\Git\Personal\ipl-player-performance\django_api
Config URL file exists: True
App URL file exists: True


In [2]:
# Preflight: verify every view module referenced by the proposed URL configuration exists.
required_files = [
    DJANGO_ROOT / "ipl_api" / "views.py",
    DJANGO_ROOT / "ipl_api" / "eda_summary_day26.py",
    DJANGO_ROOT / "ipl_api" / "eda_views.py",
    DJANGO_ROOT / "ipl_api" / "prediction_views.py",
    DJANGO_ROOT / "ipl_api" / "batch_prediction_views.py",
    DJANGO_ROOT / "ipl_api" / "dashboard_views_day28.py",
    DJANGO_ROOT / "ipl_api" / "prediction_center_views_day28.py",
    DJANGO_ROOT / "ipl_api" / "player_name_prediction_views_day28.py",
]
missing = [str(p.relative_to(PROJECT_ROOT)) for p in required_files if not p.exists()]
print("Required view modules:")
for p in required_files:
    print(("OK      " if p.exists() else "MISSING "), p.relative_to(PROJECT_ROOT))

if missing:
    raise FileNotFoundError(
        "Preflight stopped. Missing module(s): " + ", ".join(missing) +
        ". No files have been changed. Upload or restore these modules first."
    )

print("\nCurrent config/urls.py:\n")
print(CONFIG_URLS.read_text(encoding="utf-8"))
print("\nCurrent ipl_api/urls.py:\n")
print(APP_URLS.read_text(encoding="utf-8"))


Required view modules:
OK       django_api\ipl_api\views.py
OK       django_api\ipl_api\eda_summary_day26.py
OK       django_api\ipl_api\eda_views.py
OK       django_api\ipl_api\prediction_views.py
OK       django_api\ipl_api\batch_prediction_views.py
OK       django_api\ipl_api\dashboard_views_day28.py
OK       django_api\ipl_api\prediction_center_views_day28.py
OK       django_api\ipl_api\player_name_prediction_views_day28.py

Current config/urls.py:

from django.contrib import admin
from django.urls import include, path
from django.views.generic.base import RedirectView

urlpatterns = [
    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home"),
    path("api/", include("ipl_api.urls")),
    path("admin/", admin.site.urls),
]


Current ipl_api/urls.py:

from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView
from .dashboard_views_day28 import dashboard_day28
from .eda_summary_day26 import eda_summary_day26
from .eda_views i

In [3]:
# Create backups before changing either URL file.
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
config_backup = CONFIG_URLS.with_name(f"urls.py.backup_{stamp}")
app_backup = APP_URLS.with_name(f"urls.py.backup_{stamp}")
shutil.copy2(CONFIG_URLS, config_backup)
shutil.copy2(APP_URLS, app_backup)

print("Backups created:")
print(config_backup)
print(app_backup)


Backups created:
D:\Git\Personal\ipl-player-performance\django_api\config\urls.py.backup_20261023_215024
D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py.backup_20261023_215024


In [4]:
# Consolidated project-level URL configuration.
config_text = '''from django.contrib import admin
from django.urls import include, path
from django.views.generic.base import RedirectView

urlpatterns = [
    path(
        "",
        RedirectView.as_view(url="/api/dashboard/", permanent=False),
        name="home",
    ),
    path("api/", include("ipl_api.urls")),
    path("admin/", admin.site.urls),
]
'''

# Consolidated app-level URL configuration.
# Keep the more specific player-name-options route before <str:player_name>.
app_text = '''from django.urls import path

from .views import PlayerListAPIView, PlayerDetailAPIView
from .eda_summary_day26 import eda_summary_day26
from .eda_views import (
    EDABattingAPIView,
    EDABowlingAPIView,
    EDASeasonAPIView,
    EDAVenueAPIView,
    EDAOppositionAPIView,
    EDARecentFormAPIView,
)
from .prediction_views import (
    BattingPredictionAPIView,
    BowlingPredictionAPIView,
    UnifiedPredictionAPIView,
)
from .batch_prediction_views import BatchPredictionAPIView
from .dashboard_views_day28 import dashboard_day28
from .prediction_center_views_day28 import prediction_center_day28
from .player_name_prediction_views_day28 import (
    player_name_prediction_day28,
    player_name_options_day28,
    model_metrics_day28,
)

urlpatterns = [
    # Dashboard and prediction centre: unique route per path.
    path("dashboard/", dashboard_day28, name="ipl-dashboard"),
    path("predictions/", prediction_center_day28, name="ipl-predictions"),

    # Players.
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path(
        "players/search-options/",
        player_name_options_day28,
        name="player-name-options-day28",
    ),
    path(
        "players/<str:player_name>/",
        PlayerDetailAPIView.as_view(),
        name="player-detail",
    ),

    # EDA.
    path("eda/summary/", eda_summary_day26, name="eda-summary"),
    path("eda/batting/", EDABattingAPIView.as_view(), name="eda-batting"),
    path("eda/bowling/", EDABowlingAPIView.as_view(), name="eda-bowling"),
    path("eda/season/", EDASeasonAPIView.as_view(), name="eda-season"),
    path("eda/venue/", EDAVenueAPIView.as_view(), name="eda-venue"),
    path(
        "eda/opposition/",
        EDAOppositionAPIView.as_view(),
        name="eda-opposition",
    ),
    path(
        "eda/recent-form/",
        EDARecentFormAPIView.as_view(),
        name="eda-recent-form",
    ),
    path("eda/model-metrics/", model_metrics_day28, name="model-metrics"),

    # Classical ML prediction APIs.
    path("predict/", UnifiedPredictionAPIView.as_view(), name="predict-unified"),
    path(
        "predict/batting/",
        BattingPredictionAPIView.as_view(),
        name="predict-batting",
    ),
    path(
        "predict/bowling/",
        BowlingPredictionAPIView.as_view(),
        name="predict-bowling",
    ),
    path(
        "predict/batch/",
        BatchPredictionAPIView.as_view(),
        name="predict-batch",
    ),
    path(
        "predict/player/",
        player_name_prediction_day28,
        name="predict-player-name-day28",
    ),
]
'''

CONFIG_URLS.write_text(config_text, encoding="utf-8")
APP_URLS.write_text(app_text, encoding="utf-8")

print("Updated:", CONFIG_URLS)
print("Updated:", APP_URLS)
print("\nRegistered route paths:")
for line in app_text.splitlines():
    if 'path(' in line:
        print(line.strip())


Updated: D:\Git\Personal\ipl-player-performance\django_api\config\urls.py
Updated: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py

Registered route paths:
path("dashboard/", dashboard_day28, name="ipl-dashboard"),
path("predictions/", prediction_center_day28, name="ipl-predictions"),
path("players/", PlayerListAPIView.as_view(), name="player-list"),
path(
path(
path("eda/summary/", eda_summary_day26, name="eda-summary"),
path("eda/batting/", EDABattingAPIView.as_view(), name="eda-batting"),
path("eda/bowling/", EDABowlingAPIView.as_view(), name="eda-bowling"),
path("eda/season/", EDASeasonAPIView.as_view(), name="eda-season"),
path("eda/venue/", EDAVenueAPIView.as_view(), name="eda-venue"),
path(
path(
path("eda/model-metrics/", model_metrics_day28, name="model-metrics"),
path("predict/", UnifiedPredictionAPIView.as_view(), name="predict-unified"),
path(
path(
path(
path(


In [5]:
# Run Django's system check. A nonzero exit code raises an error.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "manage.py", "check"],
    cwd=DJANGO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
print("Return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Django system check failed. Read the output above before proceeding.")


System check identified no issues (0 silenced).

Return code: 0


In [6]:
# Verify URL resolution without sending HTTP requests or querying MySQL.
resolve_code = r'''
from django.urls import resolve
paths = [
    "/api/dashboard/",
    "/api/predictions/",
    "/api/players/",
    "/api/players/search-options/",
    "/api/players/search-options/",
    "/api/eda/summary/",
    "/api/eda/batting/",
    "/api/eda/bowling/",
    "/api/eda/season/",
    "/api/eda/venue/",
    "/api/eda/opposition/",
    "/api/eda/recent-form/",
    "/api/eda/model-metrics/",
    "/api/predict/",
    "/api/predict/batting/",
    "/api/predict/bowling/",
    "/api/predict/batch/",
    "/api/predict/player/",
]
for p in paths:
    match = resolve(p)
    print(f"{p:34} -> {match.url_name} [{match.func.__module__}]")
'''
result = subprocess.run(
    [sys.executable, "manage.py", "shell", "-c", resolve_code],
    cwd=DJANGO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
print("Return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("URL resolution failed. Inspect the import/traceback above.")


12 objects imported automatically (use -v 2 for details).

/api/dashboard/                    -> ipl-dashboard [ipl_api.dashboard_views_day28]
/api/predictions/                  -> ipl-predictions [ipl_api.prediction_center_views_day28]
/api/players/                      -> player-list [ipl_api.views]
/api/players/search-options/       -> player-name-options-day28 [ipl_api.player_name_prediction_views_day28]
/api/players/search-options/       -> player-name-options-day28 [ipl_api.player_name_prediction_views_day28]
/api/eda/summary/                  -> eda-summary [ipl_api.eda_summary_day26]
/api/eda/batting/                  -> eda-batting [ipl_api.eda_views]
/api/eda/bowling/                  -> eda-bowling [ipl_api.eda_views]
/api/eda/season/                   -> eda-season [ipl_api.eda_views]
/api/eda/venue/                    -> eda-venue [ipl_api.eda_views]
/api/eda/opposition/               -> eda-opposition [ipl_api.eda_views]
/api/eda/recent-form/              -> eda-recent-fo

In [7]:
# Run the existing pagination helper tests if the module exists.
test_file = DJANGO_ROOT / "ipl_api" / "tests" / "test_pagination_utils_day21.py"
if not test_file.exists():
    print("Skipped: existing pagination test file not found at", test_file)
else:
    result = subprocess.run(
        [sys.executable, "manage.py", "test",
         "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "2"],
        cwd=DJANGO_ROOT,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:\n", result.stderr)
    print("Return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Pagination regression tests failed.")


Found 6 test(s).
Skipping setup of unused database(s): default.
System check identified no issues (0 silenced).

STDERR:
 test_defaults (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_defaults) ... ok
test_paginates_records (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_paginates_records) ... ok
test_rejects_non_integer (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_rejects_non_integer) ... ok
test_rejects_oversized_page_size (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_rejects_oversized_page_size) ... ok
test_rejects_zero_page (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_rejects_zero_page) ... ok
test_valid_values (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_valid_values) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK

Return code: 0


## What this notebook does not verify

A passing system check and URL resolution confirm that Django can load the configured routes. They do **not** prove that database queries, prediction artifacts, CSRF/authentication, JavaScript, or browser workflows succeed.

Next, start the server and manually verify:
- `http://127.0.0.1:8000/api/dashboard/`
- `http://127.0.0.1:8000/api/predictions/`
- `http://127.0.0.1:8000/api/players/`

If an error occurs, save the full traceback before changing more code.

**Important:** the `/api/health/` and `/api/ingestion/...` routes are intentionally not added here because the uploaded/current URL files do not establish the exact view implementations to import. Add them only after locating or implementing those views.


## Auto-generate documentation for this notebook

The next cell generates a Markdown report inside `docs/day29/` documenting the notebook's purpose, files it edits, route inventory, commands executed, validation status, and remaining manual checks. It records the current run's observed outputs rather than claiming success in advance.

In [10]:
from pathlib import Path
import datetime
import json

DOC_DIR = PROJECT_ROOT / "docs" 
DOC_DIR.mkdir(parents=True, exist_ok=True)
report_path = DOC_DIR / "day29_url_configuration_repair_report.md"

# Capture current on-disk state and route inventory without assuming prior cells succeeded.
config_now = CONFIG_URLS.read_text(encoding="utf-8") if CONFIG_URLS.exists() else "(missing)"
app_now = APP_URLS.read_text(encoding="utf-8") if APP_URLS.exists() else "(missing)"
route_lines = [line.strip() for line in app_now.splitlines() if "path(" in line]
timestamp = datetime.datetime.now().isoformat(timespec="seconds")

report = f"""# Day 29 — Django URL Configuration Repair Report

- Generated: {timestamp}
- Repository: `{PROJECT_ROOT}`
- Notebook: `29_django_url_configuration_repair.ipynb`

## Purpose
Consolidate Django project/app URL registrations, remove duplicate root redirect and duplicate predictions path, preserve existing player/EDA/prediction routes, and verify route resolution.

## Files inspected or modified
- `django_api/config/urls.py`
- `django_api/ipl_api/urls.py`
- `django_api/ipl_api/views.py` (inspected by the developer; not modified by this notebook)
- Backup files: see the paths printed by the backup cell.

## Registered app URL patterns on disk
""" + "\n".join(f"- `{line}`" for line in route_lines) + """

## Validation
This report inventories the files currently on disk. To treat validation as passed, check the actual notebook outputs for:
- `python manage.py check`
- URL resolution for dashboard, prediction centre, players, EDA, and prediction routes
- `ipl_api.tests.test_pagination_utils_day21`

No success is assumed by this documentation cell.

## Current URL configuration — project level
```python
""" + config_now + """
```

## Current URL configuration — app level
```python
""" + app_now + """
```

## Manual follow-up
- Open `/api/dashboard/` and `/api/predictions/` in the browser.
- Confirm all desired routes resolve to the intended views.
- Add health/ingestion routes only after the corresponding views are confirmed to exist.
"""
report_path.write_text(report, encoding="utf-8")
print("Generated documentation:", report_path)
print("Bytes:", report_path.stat().st_size)


Generated documentation: D:\Git\Personal\ipl-player-performance\docs\day29_url_configuration_repair_report.md
Bytes: 5376
